In [47]:
import pandas as pd
import xml.etree.ElementTree as ET
import re

def get_full_text(elem):
    """Retourne tout le texte, y compris texte dans les sous-éléments."""
    if elem is None:
        return None
    texts = []
    if elem.text:
        texts.append(elem.text)
    for sub in elem:
        if sub.tail:
            texts.append(sub.tail)
    return ''.join(texts).strip()
tree = ET.parse("../data/JunitReport.xml")
root = tree.getroot()
rows = []

for testcase in root.findall('.//testcase'):
    test_name = testcase.get('name')

    # Récupérer contenu complet <system-out> du testcase
    testcase_system_out = get_full_text(testcase.find('system-out'))

    for step in testcase.findall('step'):
        failure_text = get_full_text(step.find('failure'))
        system_out_step_text = get_full_text(step.find('system-out'))

        step_info = {
            'test_name': test_name,
            'description': step.get('description'),
            'status': step.get('status'),
            'time': step.get('time'),
            'timestamp': step.get('timestamp'),
            'action': step.get('action'),
            'failure': failure_text,
            'system_out_testcase': testcase_system_out
        }
        rows.append(step_info)

df_steps = pd.DataFrame(rows)
df_steps

,test_name,description,status,time,timestamp,action,failure,system_out_testcase
0,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 1,PASSED,0.00,2025-06-12T06:53:13,import json,None,import json\nmain test_failed: False\nimport l...
1,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 2,PASSED,0.00,2025-06-12T06:53:13,import logging,None,import json\nmain test_failed: False\nimport l...
2,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 3,PASSED,0.00,2025-06-12T06:53:13,import WebUI.WKeywords.closeWindowUrl,None,import json\nmain test_failed: False\nimport l...
3,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 4,PASSED,0.00,2025-06-12T06:53:13,from Keywords import click,None,import json\nmain test_failed: False\nimport l...
4,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 5,PASSED,1.36,2025-06-12T06:53:15,from Keywords import CustomKeywords,None,import json\nmain test_failed: False\nimport l...
...,...,...,...,...,...,...,...,...
61553,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 25,PASSED,0.00,2025-06-11T21:32:25,from selenium.webdriver import Keys,None,import json\nmain test_failed: False\nimport l...
61554,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 26,PASSED,0.00,2025-06-11T21:32:25,findTestObject = ObjectRepository.findTestObject,None,import json\nmain test_failed: False\nimport l...
61555,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 27,PASSED,0.00,2025-06-11T21:32:25,from Keywords.util.Commands import Commands,None,import json\nmain test_failed: False\nimport l...
61556,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 28,PASSED,0.00,2025-06-11T21:32:25,from Utils.ByType import ByJavaMethod,None,import json\nmain test_failed: False\nimport l...


In [48]:
# Trouver les indices des lignes où 'status' == 'FAILED'
failed_indices = df_steps[df_steps['status'] == 'FAILED'].index

# Ajouter aussi les deux lignes précédentes de chaque ligne "FAILED"
indices_to_keep = set()
for idx in failed_indices:
    indices_to_keep.update([i for i in range(idx - 2, idx + 1) if i >= 0])

# Filtrer le DataFrame
df_filtered = df_steps.loc[sorted(indices_to_keep)].reset_index(drop=True)
df_filtered


,test_name,description,status,time,timestamp,action,failure,system_out_testcase
0,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 33,PASSED,2.62,2025-06-12T06:54:41,CustomKeywords.click.DoubleClickByColumns.clic...,None,import json\nmain test_failed: False\nimport l...
1,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 34,PASSED,1.80,2025-06-12T06:54:43,CustomKeywords.click.SelectTab.selectTab('Addr...,None,import json\nmain test_failed: False\nimport l...
2,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 35,FAILED,2.62,2025-06-12T06:54:45,while WebUI.verifyElementVisible(findTestObjec...,"Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...
3,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 33,PASSED,2.64,2025-06-12T06:56:23,CustomKeywords.click.DoubleClickByColumns.clic...,None,import json\nmain test_failed: False\nimport l...
4,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 34,PASSED,1.93,2025-06-12T06:56:25,CustomKeywords.click.SelectTab.selectTab('Call...,None,import json\nmain test_failed: False\nimport l...
...,...,...,...,...,...,...,...,...
2166,Automated Framework/Workflow/TNR/TC 60848 End ...,Step 28,PASSED,0.00,2025-06-11T21:31:16,from Utils.ByType import ByJavaMethod,None,import json\nmain test_failed: False\nimport l...
2167,Automated Framework/Workflow/TNR/TC 60848 End ...,Step 29,FAILED,32.27,2025-06-11T21:31:48,"WebUI.callTestCase('Login/LoginEnv', {'Login' ...","Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...
2168,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 27,PASSED,0.00,2025-06-11T21:32:25,from Keywords.util.Commands import Commands,None,import json\nmain test_failed: False\nimport l...
2169,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 28,PASSED,0.00,2025-06-11T21:32:25,from Utils.ByType import ByJavaMethod,None,import json\nmain test_failed: False\nimport l...


In [49]:
df_filtered['failure'] = df_filtered['failure'].apply(lambda x: "pas d'erreur" if x is None else x)

In [50]:
def extract_error_type(failure_text):
    if not isinstance(failure_text, str):
        return None

    # Regular expression to match error types
    error_pattern = re.compile(r"\b([A-Za-z_][\w\.]*?(?:Error|Exception))\b")

    matches = error_pattern.findall(failure_text)

    # Return the first match, or None if nothing found
    return matches[0] if matches else None

In [51]:
df_filtered["error_type"] = df_filtered["failure"].apply(extract_error_type)
df_filtered

,test_name,description,status,time,timestamp,action,failure,system_out_testcase,error_type
0,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 33,PASSED,2.62,2025-06-12T06:54:41,CustomKeywords.click.DoubleClickByColumns.clic...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None
1,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 34,PASSED,1.80,2025-06-12T06:54:43,CustomKeywords.click.SelectTab.selectTab('Addr...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None
2,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 35,FAILED,2.62,2025-06-12T06:54:45,while WebUI.verifyElementVisible(findTestObjec...,"Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...,TypeError
3,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 33,PASSED,2.64,2025-06-12T06:56:23,CustomKeywords.click.DoubleClickByColumns.clic...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None
4,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 34,PASSED,1.93,2025-06-12T06:56:25,CustomKeywords.click.SelectTab.selectTab('Call...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None
...,...,...,...,...,...,...,...,...,...
2166,Automated Framework/Workflow/TNR/TC 60848 End ...,Step 28,PASSED,0.00,2025-06-11T21:31:16,from Utils.ByType import ByJavaMethod,pas d'erreur,import json\nmain test_failed: False\nimport l...,None
2167,Automated Framework/Workflow/TNR/TC 60848 End ...,Step 29,FAILED,32.27,2025-06-11T21:31:48,"WebUI.callTestCase('Login/LoginEnv', {'Login' ...","Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...,Utils.Common.StepFailedException
2168,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 27,PASSED,0.00,2025-06-11T21:32:25,from Keywords.util.Commands import Commands,pas d'erreur,import json\nmain test_failed: False\nimport l...,None
2169,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 28,PASSED,0.00,2025-06-11T21:32:25,from Utils.ByType import ByJavaMethod,pas d'erreur,import json\nmain test_failed: False\nimport l...,None


In [52]:
#df_filtered.drop(columns=['failure'], inplace=True)

In [53]:
def categorize_error_type(error_type):
    if not isinstance(error_type, str):
        return "Autre"
    if error_type.endswith("Error"):
        return "Error"
    elif error_type.endswith("Exception"):
        return "Exception"
    else:
        return "Autre"  # si aucun des deux

In [54]:
df_filtered['error_category'] = df_filtered['error_type'].apply(categorize_error_type)
df_filtered

,test_name,description,status,time,timestamp,action,failure,system_out_testcase,error_type,error_category
0,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 33,PASSED,2.62,2025-06-12T06:54:41,CustomKeywords.click.DoubleClickByColumns.clic...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre
1,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 34,PASSED,1.80,2025-06-12T06:54:43,CustomKeywords.click.SelectTab.selectTab('Addr...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre
2,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 35,FAILED,2.62,2025-06-12T06:54:45,while WebUI.verifyElementVisible(findTestObjec...,"Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...,TypeError,Error
3,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 33,PASSED,2.64,2025-06-12T06:56:23,CustomKeywords.click.DoubleClickByColumns.clic...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre
4,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 34,PASSED,1.93,2025-06-12T06:56:25,CustomKeywords.click.SelectTab.selectTab('Call...,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre
...,...,...,...,...,...,...,...,...,...,...
2166,Automated Framework/Workflow/TNR/TC 60848 End ...,Step 28,PASSED,0.00,2025-06-11T21:31:16,from Utils.ByType import ByJavaMethod,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre
2167,Automated Framework/Workflow/TNR/TC 60848 End ...,Step 29,FAILED,32.27,2025-06-11T21:31:48,"WebUI.callTestCase('Login/LoginEnv', {'Login' ...","Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...,Utils.Common.StepFailedException,Exception
2168,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 27,PASSED,0.00,2025-06-11T21:32:25,from Keywords.util.Commands import Commands,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre
2169,Automated Framework/Workflow/TNR/TC 60849 End ...,Step 28,PASSED,0.00,2025-06-11T21:32:25,from Utils.ByType import ByJavaMethod,pas d'erreur,import json\nmain test_failed: False\nimport l...,None,Autre


In [55]:
df_filtered['error_type'] = df_filtered['error_type'].apply(lambda x: "pas d'erreur" if x is None else x)

In [56]:
df_filtered['error_category'].value_counts()

error_category
Autre        1546
Error         373
Exception     252
Name: count, dtype: int64

In [57]:
df_filtered.groupby('error_category')['error_type'].value_counts()

error_category  error_type                                                 
Autre           pas d'erreur                                                   1546
Error           AttributeError                                                  142
                TypeError                                                       110
                UnicodeEncodeError                                               89
                NameError                                                        12
                IndentationError                                                 11
                pyodbc.ProgrammingError                                           5
                SyntaxError                                                       3
                pyodbc.Error                                                      1
Exception       Utils.Common.StepFailedException                                 85
                selenium.common.exceptions.NoSuchElementException                63


In [58]:
error_groups = {
    error_type: group_df
    for error_type, group_df in df_filtered.groupby('error_type')
}

In [59]:
unique_errors = df_filtered['error_type'].unique()
error_type_to_id = {err: idx for idx, err in enumerate(unique_errors)}

# Appliquer ce mapping à une nouvelle colonne
df_filtered['group_id'] = df_filtered['error_type'].map(error_type_to_id)

In [60]:
df_filtered['error_type'].value_counts()

error_type
pas d'erreur                                                   1546
AttributeError                                                  142
TypeError                                                       110
UnicodeEncodeError                                               89
Utils.Common.StepFailedException                                 85
selenium.common.exceptions.NoSuchElementException                63
selenium.common.exceptions.ElementClickInterceptedException      51
selenium.common.exceptions.StaleElementReferenceException        26
TimeoutException                                                 13
NameError                                                        12
selenium.common.exceptions.ElementNotInteractableException       12
IndentationError                                                 11
pyodbc.ProgrammingError                                           5
SyntaxError                                                       3
selenium.common.exceptions.InvalidArg

In [61]:
error_counts = df_filtered['error_type'].value_counts()

# 2. Identifier les erreurs à fréquence < 11
rare_errors = error_counts[error_counts < 11].index

# 3. Remplacer dans la colonne 'error_type' par 'Autre'
df_filtered['error_type'] = df_filtered['error_type'].apply(
    lambda x: 'Autre' if x in rare_errors else x
)

In [62]:
df_filtered[['error_type', 'group_id']].sort_values('group_id')

,error_type,group_id
2132,pas d'erreur,0
2133,pas d'erreur,0
2135,pas d'erreur,0
2163,pas d'erreur,0
2156,pas d'erreur,0
...,...,...
1783,Autre,15
1720,Autre,15
1936,Autre,16
1930,Autre,16


In [63]:
unique_errors = df_filtered['error_type'].unique()
error_type_to_id = {err: idx for idx, err in enumerate(unique_errors)}

# Appliquer le mapping
df_filtered['group_id'] = df_filtered['error_type'].map(error_type_to_id)
df_filtered['error_type'].value_counts()

error_type
pas d'erreur                                                   1546
AttributeError                                                  142
TypeError                                                       110
UnicodeEncodeError                                               89
Utils.Common.StepFailedException                                 85
selenium.common.exceptions.NoSuchElementException                63
selenium.common.exceptions.ElementClickInterceptedException      51
selenium.common.exceptions.StaleElementReferenceException        26
TimeoutException                                                 13
NameError                                                        12
selenium.common.exceptions.ElementNotInteractableException       12
IndentationError                                                 11
Autre                                                            11
Name: count, dtype: int64

In [64]:
for error_type, group_id in error_type_to_id.items():
    print(f"{group_id}: {error_type}")

0: pas d'erreur
1: TypeError
2: selenium.common.exceptions.NoSuchElementException
3: Utils.Common.StepFailedException
4: selenium.common.exceptions.StaleElementReferenceException
5: UnicodeEncodeError
6: IndentationError
7: AttributeError
8: selenium.common.exceptions.ElementNotInteractableException
9: selenium.common.exceptions.ElementClickInterceptedException
10: TimeoutException
11: Autre
12: NameError


In [65]:
nb = df_filtered['error_type'].nunique()

In [66]:
df_filtered['error_type'].nunique()

13

In [67]:
df_filtered.head()

,test_name,description,status,time,timestamp,action,failure,system_out_testcase,error_type,error_category,group_id
0,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 33,PASSED,2.62,2025-06-12T06:54:41,CustomKeywords.click.DoubleClickByColumns.clic...,pas d'erreur,import json\nmain test_failed: False\nimport l...,pas d'erreur,Autre,0
1,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 34,PASSED,1.80,2025-06-12T06:54:43,CustomKeywords.click.SelectTab.selectTab('Addr...,pas d'erreur,import json\nmain test_failed: False\nimport l...,pas d'erreur,Autre,0
2,sanity/Corporate/TC-58151-Search Corporate-Add...,Step 35,FAILED,2.62,2025-06-12T06:54:45,while WebUI.verifyElementVisible(findTestObjec...,"Traceback (most recent call last):\n File ""We...",import json\nmain test_failed: False\nimport l...,TypeError,Error,1
3,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 33,PASSED,2.64,2025-06-12T06:56:23,CustomKeywords.click.DoubleClickByColumns.clic...,pas d'erreur,import json\nmain test_failed: False\nimport l...,pas d'erreur,Autre,0
4,sanity/Corporate/TC-58154-Search Corporate-Cal...,Step 34,PASSED,1.93,2025-06-12T06:56:25,CustomKeywords.click.SelectTab.selectTab('Call...,pas d'erreur,import json\nmain test_failed: False\nimport l...,pas d'erreur,Autre,0


In [68]:
df_filtered.to_csv('df_filtered.csv', index=False)

In [69]:
import pandas as pd
import html
from pathlib import Path
import plotly.express as px

OUTPUT_HTML = "../reports/report_with_chart_and_style.html"

# === Load data ===
df = df_filtered
import os
import base64
import xml.etree.ElementTree as ET
xml_path = r"C:/Users/hbakl/OneDrive/Desktop/Projet PFA/JunitReport.xml"
screenshots_dir = r"C:/Users/hbakl/OneDrive/Desktop/Projet PFA/screenshots_base64"
tree = ET.parse(xml_path)
root = tree.getroot()

import re

def load_screenshot_base64(test_name):
    """
    Finds a screenshot file matching the test case ID and returns its Base64 encoding.
    """
    # Extract TC number pattern (e.g., TC-58151)
    match = re.search(r"TC-\d+", test_name)
    if not match:
        return None
    tc_id = match.group(0)  # "TC-58151"

    for file in os.listdir(screenshots_dir):
        if file.startswith(tc_id):  # match by ID only
            with open(os.path.join(screenshots_dir, file), "rb") as img_file:
                return base64.b64encode(img_file.read()).decode("utf-8")
    return None



# Column names mapping (adjust if needed)
test_name_col = "test_name"
status_col = "status"
error_col = "error_type"
failure_col = "failure"
desc_col = "description"

# === Compute error type counts (avec toutes erreurs, y compris "pas d'erreur") ===
error_counts = df[error_col].fillna("Unknown").value_counts()

# === Préparer les données pour le barplot en excluant "pas d'erreur" ===
top5_errors = error_counts.head(5).reset_index()
top5_errors.columns = ['error_type', 'count']  # renommer clairement

# top5_errors a déjà les colonnes ['error_type', 'count']

# Ajout d'une colonne "index_erreur" (1, 2, 3, ...)
top5_errors['index_erreur'] = range(1, len(top5_errors) + 1)

# Barplot avec 'index_erreur' en x (1, 2, 3, 4, 5)
fig = px.bar(
    top5_errors,
    x='index_erreur',
    y='count',
    labels={"index_erreur": "Erreur #", "count": "Nombre d'occurrences"},
    title="Top 5 Types d'erreurs (index)",
    text='count',
    hover_data=['error_type']
)

fig.update_traces(marker_color="#4a90e2", textposition="outside")
fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=False, tickmode='linear'),
    yaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
    margin=dict(l=20, r=20, t=40, b=20),
    height=400
)

chart_html = fig.to_html(full_html=False, include_plotlyjs="cdn")

# Ensuite, pour le reste de l'interface tu peux continuer d'utiliser `error_counts` complet
# Par exemple pour les badges et sections détails, ils resteront avec "pas d'erreur".


# === Row HTML generator ===
def make_row_html(row):
    test_name_value = str(row.get(test_name_col, ""))
    status_value = str(row.get(status_col, ""))

    # Try to load screenshot if failed
    img_tag = ""
    if status_value.upper() == "FAILED":
        img_b64 = load_screenshot_base64(test_name_value)
        if img_b64:
            img_tag = f"""
            <div style='margin-top:8px; text-align:center;'>
                <img src="data:image/png;base64,{img_b64}"
                     alt="Screenshot" style="max-width:600px; border:1px solid #ccc;"/>
            </div>
            """

    return f"""
    <tr>
      <td>{html.escape(test_name_value)}</td>
      <td>{html.escape(status_value)}</td>
      <td>{html.escape(str(row.get(error_col, "")))}</td>
      <td>
        <pre style="white-space:pre-wrap;margin:0;">{html.escape(str(row.get(failure_col, "")))}</pre>
        {img_tag}
      </td>
      <td>{html.escape(str(row.get(desc_col, "")))}</td>
    </tr>
    """

# === CSS Styling with boxes and modern look ===
style_block = """
<style>
body {
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    margin: 20px;
    background-color: #f8f9fa;
    color: #333;
}
h1 {
    text-align: center;
    color: #2c3e50;
    margin-bottom: 30px;
}
.card {
    padding: 20px;
    border-radius: 12px;
    background: white;
    box-shadow: 0 2px 10px rgba(0,0,0,0.1);
    margin-bottom: 20px;
}

.badge {
    background: #4a90e2;
    color: white;
    padding: 6px 12px;
    border-radius: 999px;
    font-size: 13px;
    margin-right: 8px;
    display: inline-block;
    margin-bottom: 10px;
}

.error-box {
    background: white;
    border: 1px solid #ddd;
    border-left: 6px solid #4a90e2;
    border-radius: 10px;
    padding: 15px 20px;
    margin-bottom: 25px;
    box-shadow: 0 4px 10px rgba(74, 144, 226, 0.1);
}

.error-box summary {
    cursor: pointer;
    font-weight: 700;
    font-size: 1.1em;
    padding: 10px 0;
    user-select: none;
}

.error-box details[open] summary {
    color: #1a73e8;
}

table {
    border-collapse: collapse;
    width: 100%;
    margin-top: 12px;
    font-size: 14px;
}
th, td {
    border: 1px solid #dee2e6;
    padding: 10px 12px;
    text-align: left;
    vertical-align: top;
}
th {
    background-color: #f1f3f5;
}
pre {
    font-family: 'Segoe UI Mono', Consolas, monospace;
    font-size: 13px;
    margin: 0;
}
</style>
"""

# === Build HTML ===
html_parts = [
    "<!DOCTYPE html><html lang='en'><head><meta charset='utf-8'/>",
    "<title>Test Failures Dashboard</title>",
    style_block,
    "</head><body>",
    "<h1>Test Failures Dashboard</h1>",
    f"<div class='card'>{chart_html}</div>",
]

# Error type badges
html_parts.append("<div style='margin-bottom: 20px;'>")
for et, cnt in error_counts.items():
    html_parts.append(f"<span class='badge'>{html.escape(str(et))} — {cnt}</span>")
html_parts.append("</div>")

# Collapsible details for each error type in box
for et, cnt in error_counts.items():
    safe_et_id = html.escape(str(et)).replace(" ", "_").replace(".", "_")
    subset = df[df[error_col].fillna("Unknown") == et]
    html_parts.append(f"<details id='{safe_et_id}' class='error-box'>")
    html_parts.append(f"<summary>{html.escape(str(et))} — {cnt} test{'s' if cnt>1 else ''}</summary>")
    html_parts.append("<table><thead><tr><th>Test Name</th><th>Status</th><th>Error Type</th><th>Failure Message</th><th>Description</th></tr></thead><tbody>")
    for _, row in subset.iterrows():
        html_parts.append(make_row_html(row))
    html_parts.append("</tbody></table></details>")

# JS to make chart bars clickable
html_parts.append("""
<script>
document.addEventListener('DOMContentLoaded', function(){
    document.querySelectorAll('.xtick text').forEach(t => {
        t.style.cursor = 'pointer';
        t.addEventListener('click', () => {
            const id = t.textContent.replace(/\\s+/g, '_').replace(/\\./g, '_');
            const el = document.getElementById(id);
            if(el){
                el.open = true;
                el.scrollIntoView({behavior: 'smooth'});
            }
        });
    });
});
</script>
""")

html_parts.append("</body></html>")

# === Save HTML file ===
Path(OUTPUT_HTML).write_text("\n".join(html_parts), encoding="utf-8")
print(f"HTML report saved to {OUTPUT_HTML}")

HTML report saved to ../reports/report_with_chart_and_style.html
